# 第5回 今日の値と次の日の値


第4回では、ニューヨークの正午の値が東京の17時の値より円安だった日に、次の日の東京も円安になりやすいことを見た。2024年には両方に値がある236日のうち181日、約77%の日で向きがそろった。

ただし、ニューヨークの正午の値が分かるのは東京の17時の8〜9時間後である。この回では東京の値だけを使い、今日までの値から次の日の値がどこまで分かるかを調べる。レートの値そのものと、前の日からの動きに分けて調べていこう。


## 実行の準備

`finance_data.zip` をNotebookと同じフォルダに置き、下のセルを実行する。セルは上から順に実行すること。「変数が見つからない」というエラーが出たら、前のセルを実行し忘れている。


In [ ]:
from pathlib import Path
from zipfile import ZipFile

if not Path("course_data.py").exists():
    with ZipFile("finance_data.zip") as archive:
        archive.extractall(".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_data import (
    load_boj_fx, load_boj_monthly, load_fed_fx, load_fed_rates,
    configure_plotting, format_axis,
)

configure_plotting()
pd.options.display.float_format = "{:.3f}".format


## ドル・円レートの時系列

日銀の表を読み込み、`.dropna()` で値のない日を除いて `rate` を作る。第4回の「次の日の東京の動き」の節で作った `rate` と同じもので、東京の17時の値が観測日の順に並んでいる。


In [ ]:
boj = load_boj_fx()
rate = boj["usd_jpy"].dropna()
print(f"観測日：{len(rate)} 日")
rate.head()


2000〜2025年の6372日分の値がある。このように時間の順に並んだ観測値の列を時系列という。26年分の時系列を図に描いてみよう。


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rate.index, rate, linewidth=0.8)
ax.set(title="ドル・円（東京17時）", ylabel="円／ドル")
format_axis(ax)
fig.tight_layout()
plt.show()


レートは2000年代の後半から下がり、2011〜2012年には80円を下回った。2022年からは急に上がり、150円を超えている。最も低い日と最も高い日を確かめておく。


In [ ]:
print(f"最も低い値：{rate.min():.2f} 円／ドル（{rate.idxmin().date()}）")
print(f"最も高い値：{rate.max():.2f} 円／ドル（{rate.idxmax().date()}）")


最も低いのは2011年10月28日の75.84円、最も高いのは2024年7月3日の161.77円で、高い値は低い値の2倍を超える。この回では、このようなレートの値そのものを水準、前の観測日から水準がどれだけ動いたかを変化と呼んで、2つを区別しよう。


## 前日の値の列

第4回では `rate.shift(-1)` で値を1つ上にずらし、各行に次の観測日の値を入れた。引数を `1` にした `rate.shift(1)` では値が1つ下にずれ、各行には前の観測日の値が入る。土日や祝日は除いてあるので、月曜日の行に入るのは金曜日の値である。ここからは前の観測日を「前日」と呼ぶ。

`pd.DataFrame({ })` で前日の値と当日の値を並べた表を作り、`levels` という名前を付ける。


In [ ]:
levels = pd.DataFrame({
    "前日の値": rate.shift(1),
    "当日の値": rate,
})
levels.loc["2024-01-04":"2024-01-12"]


2024年1月4日の行の前日の値141.40円は、年末の最後の観測日である2023年12月29日の値である。1月9日（火曜日）の行には、土日と成人の日をとばして1月5日（金曜日）の値145.07円が入っている。

各行には連続する2つの観測日の値が並ぶ。前日を今日、当日を次の日と読みかえれば、今日の値で次の日の値がどこまで分かるかをこの表で調べられる。

表の最初の行も見ておこう。


In [ ]:
levels.head(3)


最初の行の2000年1月4日には前の観測日がないので、前日の値は `NaN` になっている。この行は計算に使えないので、`.dropna()` で除いておく。


In [ ]:
levels = levels.dropna()
print(f"前日の値と当日の値がそろう日：{len(levels)} 日")


## 前日の値と当日の値の散布図

前日の値と当日の値の関係を図で確かめよう。横軸に前日の値、縦軸に当日の値をとって、1日を1つの点で表す。このように2つの値の組を点で描いた図を散布図という。

第4回で発表日に赤い点を打ったときと同じく、`ax.plot()` に `"o"` を渡せば点が描ける。6371日分の点を描くので、`markersize=2` で点を小さくしておこう。

- 灰色の線は (70, 70) と (170, 170) の2点を結んだ線で、この線の上にある点は前日から値が動かなかった日を表す。
- 横軸が日付ではないので、`format_axis()` には `dates=False` を付ける。


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(levels["前日の値"], levels["当日の値"], "o", markersize=2)
ax.plot([70, 170], [70, 170], color="gray", linewidth=0.8)
ax.set(xlabel="前日の値（円／ドル）", ylabel="当日の値（円／ドル）",
       title="前日の値と当日の値（2000〜2025年）")
format_axis(ax, dates=False)
fig.tight_layout()
plt.show()


点はすべて灰色の線のすぐそばに並んでいる。当日の値は前日の値からあまり離れないので、前日の値が分かれば当日の値もおおよそ分かる。


## 相関係数

点が右上がりの直線にどれだけ近く並ぶかは、相関係数という数で表せる。相関係数がとる値は −1 から 1 までで、点が右上がりの直線にぴったり並べば1、右下がりの直線にぴったり並べば −1 になる。点が散らばって右上がりとも右下がりともいえないときは0に近い。

`列1.corr(列2)` は2つの列の相関係数を計算する。


In [ ]:
level_corr = levels["前日の値"].corr(levels["当日の値"])
print(f"前日の値と当日の値の相関係数：{level_corr:.3f}")


相関係数は0.999で、1にきわめて近い。水準は前日の値でほぼ決まるといってよい。

ただし、相関係数がここまで1に近くなるのは、26年の間にレートが76円から162円まで動き、それに比べて1日の動きが小さいからでもある。期間を2024年の1年に絞って、同じ散布図を描いてみよう。灰色の線は (138, 138) と (164, 164) を結ぶ。


In [ ]:
levels_2024 = levels.loc["2024"]

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(levels_2024["前日の値"], levels_2024["当日の値"], "o", markersize=3)
ax.plot([138, 164], [138, 164], color="gray", linewidth=0.8)
ax.set(xlabel="前日の値（円／ドル）", ylabel="当日の値（円／ドル）",
       title="前日の値と当日の値（2024年）")
format_axis(ax, dates=False)
fig.tight_layout()
plt.show()


2024年のレートは140円台から160円台の間で動いた。26年分の図より軸の範囲が狭いので、点が灰色の線から離れているのが分かる。相関係数も求めておく。


In [ ]:
level_corr_2024 = levels_2024["前日の値"].corr(levels_2024["当日の値"])
print(f"2024年の前日の値と当日の値の相関係数：{level_corr_2024:.3f}")


2024年だけでは0.980で、26年分の0.999より小さい。1年の間にレートが動いた幅は26年より狭く、それに比べると1日の動きが大きいので、相関係数が下がった。それでも当日の値は前日の値の近くにあった。

散布図で灰色の線より上にある点は前日より円安になった日、下にある点は円高になった日である。点が線から縦にどれだけ離れているかは、前日から値がどれだけ動いたかを表す。次はこの前日からの変化を取り出して調べよう。


## 前日からの変化

当日の値から前日の値を引くと、前日からの変化が求まる。`rate - rate.shift(1)` と書いてもよいが、同じ計算は `rate.diff()` と短く書ける。

`.diff()` は各行の値から1つ前の行の値を引く関数で、こうして求めた変化を差分ともいう。`diff` は英語の difference（差）からとった名前である。

2通りの書き方の結果を並べて、同じ値になるかを確かめよう。


In [ ]:
change = rate.diff()

pd.DataFrame({
    "当日の値": rate,
    "前日の値": rate.shift(1),
    "引き算": rate - rate.shift(1),
    "rate.diff()": change,
}).loc["2024-01-04":"2024-01-12"]


右の2列は同じ値になっている。1月4日は前日（12月29日）より1.98円の円安、1月9日は1.00円の円高だった。変化がプラスなら円安・ドル高、マイナスなら円高・ドル安である。

`change` の最初の行も確かめておく。


In [ ]:
change.head(3)


2000年1月4日の行には1つ前の行がないので、変化は `NaN` になる。2行目の0.200は1月5日の103.19円から1月4日の102.99円を引いた値である。

26年分の変化を図に描く。


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(change.index, change, linewidth=0.5)
ax.axhline(0, color="gray", linewidth=0.8)
ax.set(title="前日からの変化（東京17時）", ylabel="円／ドル")
format_axis(ax)
fig.tight_layout()
plt.show()


変化の線は0の上下を行き来し、多くの日は±1円の幅に収まる。ところどころに3円を超える日がある。振れ幅が大きい時期と小さい時期があることも分かる。


## 前日の変化と当日の変化

変化についても前日と当日を同じ行に並べる。`change.shift(1)` で前日の変化を同じ行に入れ、`.dropna()` で `NaN` のある最初の2行を除く。


In [ ]:
changes = pd.DataFrame({
    "前日の変化": change.shift(1),
    "当日の変化": change,
}).dropna()
changes.loc["2024-01-04":"2024-01-12"]


1月5日の行の前日の変化1.98円は、1月4日の行の当日の変化である。1月4日と5日は2日続けて円安になり、1月9日は円安の次の日に円高へ向きが変わった。

前日の変化を横軸、当日の変化を縦軸にとって散布図を描こう。`ax.axvline(0)` は `ax.axhline(0)` の縦向きにあたり、横軸の0の位置に縦線を引く。2本の線で分けた右上の区画は2日続けて円安になった日、左下の区画は2日続けて円高になった日を表す。


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(changes["前日の変化"], changes["当日の変化"], "o", markersize=2)
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)
ax.set(xlabel="前日の変化（円／ドル）", ylabel="当日の変化（円／ドル）",
       title="前日の変化と当日の変化（2000〜2025年）")
format_axis(ax, dates=False)
fig.tight_layout()
plt.show()


点は原点を中心に丸く散らばっている。前日に円安だった日（縦線の右側）だけを見ても、点は横線の上と下に同じくらいある。相関係数で確かめよう。


In [ ]:
change_corr = changes["前日の変化"].corr(changes["当日の変化"])
print(f"前日の変化と当日の変化の相関係数：{change_corr:.3f}")


相関係数は −0.019 で、ほぼ0である。水準の0.999とは大きく違い、前日の変化と当日の変化の間には直線的な関係がほとんどない。


## 変化の向きがそろう日の割合

第4回と同じように向きだけを取り出し、2024年について数える。前日と当日のそれぞれについて円安になったかを `True`／`False` で表し、`pd.crosstab()` で組合せごとの日数を数える。変化が0の日は、第4回と同じく「円高・変化なし」の側に入れる。


In [ ]:
changes_2024 = changes.loc["2024"]
prev_weak = changes_2024["前日の変化"] > 0   # 前日に円安なら True
today_weak = changes_2024["当日の変化"] > 0  # 当日に円安なら True

pd.DataFrame({
    "前日に円安": prev_weak,
    "当日に円安": today_weak,
}).head()


1月4日と5日は2つとも `True` で、向きがそろっている。1月9日は前日が円安、当日が円高なので、向きがそろっていない。組合せごとの日数を `pd.crosstab()` で数え、行と列の見出しを日本語に付け替える。


In [ ]:
table = pd.crosstab(prev_weak, today_weak)
table.index = ["前日は円高・変化なし", "前日は円安"]
table.columns = ["当日は円高・変化なし", "当日は円安"]
table


向きがそろったのは、前日も当日も円高・変化なしだった39日（左上）と、前日も当日も円安だった74日（右下）である。前日に円安だった140日のうち、当日も円安だったのは74日で、残りの66日は円高・変化なしに変わった。

割合は `.mean()` で求められる。`True` を1、`False` を0として平均をとると、`True` の割合になるからである。`prev_weak == today_weak` は2つの向きが同じ日に `True` になる。


In [ ]:
same_2024 = (prev_weak == today_weak).mean()
print(f"向きがそろった日の割合（2024年）：{100 * same_2024:.1f}%")


2024年に向きがそろった日は46.1%で、半分に届かない。第4回でニューヨークの値を手がかりにしたときの約77%とは大きな開きがある。前日と同じ向きを当日の予想にしていたら、2024年は半分以上の日で外れたことになる。

1年分では偶然の偏りもありうるので、26年分でも数えてみよう。


In [ ]:
same_all = ((changes["前日の変化"] > 0) == (changes["当日の変化"] > 0)).mean()
print(f"向きがそろった日の割合（2000〜2025年）：{100 * same_all:.1f}%")


26年分では49.9%だった。硬貨を投げて表なら円安、裏なら円高と予想しても、当たる割合は50%ほどになる。前日の向きを使った予想は、硬貨で決めた予想とほとんど変わらない。東京の値では、前日の変化から次の日の向きは読めない。


## 新しい情報とレートの動き

前日の向きから次の日の向きが読めないことには、市場の仕組みから説明が付く。

今日のレートが150円で、明日は151円になるとだれもが分かっていたとしよう。今日150円でドルを買って明日151円で売れば、1ドルにつき1円の利益になる。

そう考えた人が今日のうちにドルを買うので、第4回で説明したようにレートは上がる。151円より安いうちは買えば利益が出るから、買いはレートが151円近くになるまで続く。明日の値上がりは、今日のうちにほとんど済んでしまう。

前日の変化はだれでも見られる値である。そこから次の日の動きが分かるなら、その動きも同じように前の日のうちにレートに表れてしまう。次の日のレートを動かすのは、雇用統計の数字のように前の日にはまだ分からなかった新しい情報だと考えられる。

第4回でニューヨークの値が次の日の東京の向きをよく当てたことも、同じ見方で説明できる。ニューヨークの正午の値には、東京の17時より後に出た新しい情報がすでに入っていた。東京の17時の時点で分かっている値だけを使うと、向きが当たる割合は半分ほどになる。


## 変化の大きさ

向きとは別に、変化の大きさも調べておこう。変化の図では、上下の振れ幅が大きい時期と小さい時期があった。`.abs()` で符号を外した変化の大きさについて、前日と当日の相関係数を求める。


In [ ]:
size_corr = changes["前日の変化"].abs().corr(changes["当日の変化"].abs())
print(f"前日と当日の変化の大きさの相関係数：{size_corr:.3f}")


相関係数は0.166で、変化そのものの −0.019 より大きく、水準の0.999よりはずっと小さい。大きく動いた日の次の日は、円安か円高かは分からないが、また大きく動きやすい。変化の大きさが続く性質は、後の回で変動の大きさを測るときにもう一度調べる。


## 今日の値による次の日の予想

水準は前日の値でほぼ決まり、変化の向きは前日の変化から読めなかった。そこで、今日の値をそのまま次の日の予想にして、この予想がどれだけ外れるかを測ろう。当日の値から予想（前日の値）を引いた差を、予想の誤差と呼ぶ。


In [ ]:
forecast = pd.DataFrame({
    "予想（前日の値）": rate.shift(1),
    "当日の値": rate,
})
forecast["誤差"] = forecast["当日の値"] - forecast["予想（前日の値）"]
forecast.loc["2024-01-04":"2024-01-12"]


誤差の列は、`rate.diff()` で求めた変化と同じ値になっている。今日の値を次の日の予想にしたときの誤差は、次の日の変化そのものである。2024年の誤差を取り出し、まずそのまま平均してみる。


In [ ]:
error_2024 = forecast.loc["2024"]["誤差"]
print(f"2024年の誤差の平均：{error_2024.mean():.2f} 円／ドル")


誤差の平均は0.07円で、0に近い。プラスの誤差とマイナスの誤差が打ち消し合うので、この値は予想がどれだけ外れたかを表していない。外れの大きさを測るには、`.abs()` で誤差の大きさにしてから平均する。


In [ ]:
print(f"2024年の誤差の大きさの平均：{error_2024.abs().mean():.2f} 円／ドル（{len(error_2024)} 日）")


2024年に今日の値を次の日の予想にすると、誤差の大きさの平均は0.70円だった。1ドルあたり0.70円の誤差は、1000ドルを両替する人にとっては700円ほどの違いになる。後の回でほかの予想の方法を学んだら、この0.70円より外れが小さくなるかどうかで比べる。

誤差が1円以内に収まった日と、最も大きく外れた日も確かめる。


In [ ]:
within_1yen = error_2024.abs() <= 1
print(f"誤差が1円以内の日：{within_1yen.sum()} 日（{100 * within_1yen.mean():.1f}%）")
print(f"最も大きな誤差：{error_2024.abs().max():.2f} 円／ドル（{error_2024.abs().idxmax().date()}）")


4日のうち3日ほどは1円以内に収まったが、2024年8月5日には5.72円外れた。誤差の大きさは日によって大きく違う。

同じ計算を2023〜2025年の各年について行う。第3回で練習した `for` の繰り返しを使う。


In [ ]:
for year in ["2023", "2024", "2025"]:
    error_year = forecast.loc[year]["誤差"]
    print(f"{year}年：{error_year.abs().mean():.2f} 円／ドル")


2023〜2025年の誤差の大きさの平均は0.67〜0.70円で、3年ともほぼ同じだった。演習1の追加問題では、レートが80円前後だった2012年について同じ計算をする。


## 演習1：ニューヨークの値と平均による予想

1. 問1の計算欄では、FRBのニューヨーク正午の値について、2024年に前日の変化と当日の変化の向きがそろった日の割合を、本文と同じ手順で求めている。セルを実行して表示された割合を、東京の値の46.1%と比べよ。
2. 第4回で見たように、日銀の表には土日と祝日の行もあり、その日の値は `NaN` になっている。この表で `.dropna()` をせずに `.diff()` を計算すると、2024年1月9日（火曜日）の行の変化はどうなるか。この回で `.diff()` の前に `.dropna()` をした理由を説明せよ。
3. 次の日の値を予想する単純な方法には、今日の値をそのまま使う方法のほかに、過去の値の平均を使う方法がある。2000〜2023年のレートの平均を2024年のすべての日の予想にしたときの、誤差の大きさの平均を求めよ。今日の値を使う予想の0.70円と比べて、どちらの方法がよいかを理由とともに答えよ。

追加問題：レートが80円前後だった2012年について、今日の値を次の日の予想にしたときの誤差の大きさの平均を求めよ。また、2012年と2024年のレートの平均を求めよ。誤差の大きさの平均とレートの平均のそれぞれについて、2012年の値が2024年の値の何倍かを計算し、2012年の誤差が小さかったことをレートの水準が低かったことだけで説明できるかを答えよ。

問3と追加問題では、本文のセルを参考にしてコードを書く。


In [ ]:
# 問1の計算欄：ニューヨークの値で、本文と同じ計算をする
fed = load_fed_fx()
ny_rate = fed["usd_jpy"].dropna()
ny_change = ny_rate.diff()
ny_changes = pd.DataFrame({
    "前日の変化": ny_change.shift(1),
    "当日の変化": ny_change,
}).dropna()
ny_2024 = ny_changes.loc["2024"]
same_ny = ((ny_2024["前日の変化"] > 0) == (ny_2024["当日の変化"] > 0)).mean()
print(f"向きがそろった日の割合（NY、2024年）：{100 * same_ny:.1f}%（{len(ny_2024)} 日）")


In [ ]:
# 問3の計算欄
# 2000〜2023年のレートは rate.loc[:"2023"]、2024年のレートは rate.loc["2024"] で取り出せる。


In [ ]:
# 追加問題の計算欄
# 誤差は forecast.loc["2012"]["誤差"]、レートは rate.loc["2012"] で取り出せる。


### 演習1の記入欄

- 問1：ニューヨークの値で向きがそろった日の割合と、東京の値との比較：
- 問2：1月9日の行の変化がどうなるか、先に `.dropna()` をした理由：
- 問3：平均による予想の誤差の大きさの平均、0.70円との比較、どちらの方法がよいかとその理由：
- 追加問題：2012年の誤差の大きさの平均、2012年と2024年のレートの平均、2つの倍率、水準だけで説明できるか：


<details>
<summary>演習1のヒント</summary>

問1では、ニューヨークの割合が東京の46.1%と大きく違うか、同じくらいかを見よう。

問2では、日銀の表で1月9日のすぐ上の行が1月8日（成人の日）であることに注目しよう。`NaN` を使った引き算の結果は `NaN` になる。

問3の `rate.loc[:"2023"]` は、2023年までのすべての行を取り出す書き方である。その平均を `.mean()` で求め、2024年のレートから引いた差を誤差とする。誤差の大きさの平均は、本文で2024年について求めたときと同じく `.abs().mean()` で求める。

追加問題の誤差の大きさの平均は、「今日の値による次の日の予想」の節で2024年について求めたセルと同じ形で書ける。倍率は2012年の値を2024年の値で割って求める。誤差がレートの水準に比例して小さくなるのなら、2つの倍率はほぼ同じになるはずである。

</details>


## 出典

[日本銀行「外国為替市況」](https://www.boj.or.jp/statistics/market/forex/fxdaily/index.htm)の東京市場ドル・円スポット、17時時点（円／ドル）を使用。配布データの対象期間は2000〜2025年である。

演習1で使う米国の為替レートは [FRB H.10（為替）](https://www.federalreserve.gov/releases/h10/)による。
